# Data Quality Notebook 02: Validation report notebook

This notebook shows how to turn data quality checks into a clear validation report.

The goal is to move from individual checks to a reusable report that someone can read after a notebook, batch job, or Airflow task finishes.

## Notebook objectives

By the end of this notebook, you should be able to:

- define common data quality expectations
- validate schema, nulls, uniqueness, accepted values, numeric ranges, and freshness
- collect validation results in a standard format
- separate high severity checks from lower severity checks
- summarize failed checks for investigation

## 1. Why validation reports matter

In data engineering, it is not enough to run a check in your head or inspect a few rows manually.

A validation report gives us a repeatable way to answer:

- What did we check?
- Did each check pass or fail?
- Which columns or rows are affected?
- Is the issue severe enough to block the pipeline?
- What should we investigate next?

In [ ]:
import pandas as pd

## 2. Create a small dataset with intentional issues

This dataset represents a target order table after a load. It has a few problems on purpose:

- a duplicated `order_id`
- a missing `order_id`
- a missing `customer_id`
- an invalid `status`
- a negative amount
- an unexpected currency
- an invalid date string

In [ ]:
orders_df = pd.DataFrame(
    {
        "order_id": [1001, 1002, 1003, 1003, 1005, 1006, None, 1008],
        "customer_id": ["C001", "C002", "C003", "C003", "C005", None, "C007", "C008"],
        "order_date": [
            "2026-05-01",
            "2026-05-02",
            "2026-05-03",
            "2026-05-03",
            "2026-05-04",
            "2026-04-15",
            "2026-05-05",
            "not_a_date",
        ],
        "status": ["complete", "complete", "refunded", "refunded", "cancelled", "pending", "complete", "complete"],
        "net_amount": [120.0, 70.0, 180.0, 180.0, 0.0, 55.0, -10.0, 40.0],
        "currency": ["USD", "USD", "USD", "USD", "USD", "USD", "EUR", "USD"],
    }
)

orders_df["order_date"] = pd.to_datetime(orders_df["order_date"], errors="coerce")

orders_df

## 3. Design the report shape

Before writing checks, decide what every result should contain.

In this notebook, every check returns:

- `check_name`: the name of the validation
- `column`: the column being checked, or `table` for table-level checks
- `expectation`: what should be true
- `observed_value`: what we found
- `failed_rows`: how many rows failed
- `severity`: how important the check is
- `passed`: `True` or `False`
- `sample_rows`: a few row indexes to inspect

In [ ]:
def make_result(
    check_name,
    column,
    expectation,
    observed_value,
    failed_rows,
    severity,
    passed,
    sample_rows=None,
):
    """Create one validation result row."""
    return {
        "check_name": check_name,
        "column": column,
        "expectation": expectation,
        "observed_value": observed_value,
        "failed_rows": failed_rows,
        "severity": severity,
        "passed": passed,
        "sample_rows": sample_rows or [],
    }

## 4. Define validation expectations

A useful pattern is to define expectations in one place. Later, this could come from a YAML file or a job configuration.

In [ ]:
validation_config = {
    "expected_columns": ["order_id", "customer_id", "order_date", "status", "net_amount", "currency"],
    "not_null": ["order_id", "customer_id", "order_date", "status", "net_amount"],
    "unique": ["order_id"],
    "accepted_values": {
        "status": ["complete", "cancelled", "refunded"],
        "currency": ["USD"],
    },
    "numeric_ranges": {
        "net_amount": {"min_value": 0, "max_value": None},
    },
    "freshness": {
        "order_date": "2026-05-01",
    },
}

## 5. Validate schema

Schema checks confirm that required columns exist and that no unexpected columns appeared.

This is especially important when upstream files, source tables, or connector payloads change.

In [ ]:
def validate_schema(df, expected_columns, severity="high"):
    actual_columns = list(df.columns)
    missing_columns = sorted(set(expected_columns) - set(actual_columns))
    unexpected_columns = sorted(set(actual_columns) - set(expected_columns))
    passed = not missing_columns and not unexpected_columns

    return make_result(
        check_name="schema_columns",
        column="table",
        expectation=f"columns match expected schema: {expected_columns}",
        observed_value={"missing": missing_columns, "unexpected": unexpected_columns},
        failed_rows=len(missing_columns) + len(unexpected_columns),
        severity=severity,
        passed=passed,
    )


pd.DataFrame([validate_schema(orders_df, validation_config["expected_columns"])])

## 6. Validate required values

Not-null checks are common for primary keys, foreign keys, dates, status columns, and measures.

In [ ]:
def validate_not_null(df, column, severity="high"):
    failed_mask = df[column].isna()
    failed_indexes = df[failed_mask].index.tolist()

    return make_result(
        check_name="not_null",
        column=column,
        expectation="value is not null",
        observed_value=int(failed_mask.sum()),
        failed_rows=int(failed_mask.sum()),
        severity=severity,
        passed=failed_mask.sum() == 0,
        sample_rows=failed_indexes[:5],
    )


not_null_results = [validate_not_null(orders_df, column) for column in validation_config["not_null"]]

pd.DataFrame(not_null_results)

## 7. Validate uniqueness

Uniqueness checks help catch duplicated primary keys or duplicated business keys.

Here we check whether `order_id` is unique when it is present. The not-null check handles missing values separately.

In [ ]:
def validate_unique(df, column, severity="high"):
    failed_mask = df[column].notna() & df[column].duplicated(keep=False)
    failed_indexes = df[failed_mask].index.tolist()

    return make_result(
        check_name="unique",
        column=column,
        expectation="non-null values are unique",
        observed_value=int(failed_mask.sum()),
        failed_rows=int(failed_mask.sum()),
        severity=severity,
        passed=failed_mask.sum() == 0,
        sample_rows=failed_indexes[:5],
    )


unique_results = [validate_unique(orders_df, column) for column in validation_config["unique"]]

pd.DataFrame(unique_results)

## 8. Validate accepted values

Accepted-value checks are useful for status columns, region codes, currencies, flags, and other categorical fields.

In [ ]:
def validate_accepted_values(df, column, accepted_values, severity="medium"):
    failed_mask = df[column].notna() & ~df[column].isin(accepted_values)
    failed_indexes = df[failed_mask].index.tolist()

    return make_result(
        check_name="accepted_values",
        column=column,
        expectation=f"value is one of {accepted_values}",
        observed_value=sorted(df.loc[failed_mask, column].dropna().unique().tolist()),
        failed_rows=int(failed_mask.sum()),
        severity=severity,
        passed=failed_mask.sum() == 0,
        sample_rows=failed_indexes[:5],
    )


accepted_value_results = [
    validate_accepted_values(orders_df, column, accepted_values)
    for column, accepted_values in validation_config["accepted_values"].items()
]

pd.DataFrame(accepted_value_results)

## 9. Validate numeric ranges

Numeric range checks catch impossible or suspicious values.

For example, an order amount should usually not be negative.

In [ ]:
def validate_numeric_range(df, column, min_value=None, max_value=None, severity="medium"):
    failed_mask = pd.Series(False, index=df.index)

    if min_value is not None:
        failed_mask = failed_mask | (df[column] < min_value)

    if max_value is not None:
        failed_mask = failed_mask | (df[column] > max_value)

    failed_mask = failed_mask & df[column].notna()
    failed_indexes = df[failed_mask].index.tolist()

    return make_result(
        check_name="numeric_range",
        column=column,
        expectation=f"value between {min_value} and {max_value}",
        observed_value=int(failed_mask.sum()),
        failed_rows=int(failed_mask.sum()),
        severity=severity,
        passed=failed_mask.sum() == 0,
        sample_rows=failed_indexes[:5],
    )


numeric_range_results = [
    validate_numeric_range(
        orders_df,
        column,
        min_value=rules.get("min_value"),
        max_value=rules.get("max_value"),
    )
    for column, rules in validation_config["numeric_ranges"].items()
]

pd.DataFrame(numeric_range_results)

## 10. Validate freshness

Freshness checks confirm that the table contains data recent enough for the workflow.

This example checks whether the latest `order_date` is at least `2026-05-01`.

In [ ]:
def validate_freshness(df, date_column, minimum_latest_date, severity="high"):
    minimum_latest_date = pd.to_datetime(minimum_latest_date)
    latest_date = df[date_column].max()
    passed = pd.notna(latest_date) and latest_date >= minimum_latest_date

    return make_result(
        check_name="freshness",
        column=date_column,
        expectation=f"latest date is at least {minimum_latest_date.date()}",
        observed_value=None if pd.isna(latest_date) else latest_date.date(),
        failed_rows=0 if passed else len(df),
        severity=severity,
        passed=passed,
    )


freshness_results = [
    validate_freshness(orders_df, column, minimum_latest_date)
    for column, minimum_latest_date in validation_config["freshness"].items()
]

pd.DataFrame(freshness_results)

## 11. Build the complete validation report

Now we combine all checks into one report.

In [ ]:
validation_results = []
validation_results.append(validate_schema(orders_df, validation_config["expected_columns"]))
validation_results.extend(not_null_results)
validation_results.extend(unique_results)
validation_results.extend(accepted_value_results)
validation_results.extend(numeric_range_results)
validation_results.extend(freshness_results)

validation_report = pd.DataFrame(validation_results)
validation_report["status"] = validation_report["passed"].map({True: "PASS", False: "FAIL"})

validation_report

## 12. Summarize failures

A report is most useful when it makes failures easy to find.

In [ ]:
failure_summary = (
    validation_report.groupby(["status", "severity"], as_index=False)
    .size()
    .rename(columns={"size": "check_count"})
    .sort_values(["status", "severity"])
)

failure_summary

In [ ]:
failed_check_details = validation_report.loc[
    ~validation_report["passed"],
    ["check_name", "column", "severity", "failed_rows", "observed_value", "sample_rows"],
].sort_values(["severity", "check_name"])

failed_check_details

## 13. Print a simple human-readable summary

This kind of summary is useful in notebook output, logs, or Airflow task logs.

In [ ]:
def print_report_summary(report):
    total_checks = len(report)
    passed_checks = int(report["passed"].sum())
    failed_checks = total_checks - passed_checks

    print(f"Total checks: {total_checks}")
    print(f"Passed checks: {passed_checks}")
    print(f"Failed checks: {failed_checks}")

    if failed_checks > 0:
        print("\nFailed checks:")
        for _, row in report[~report["passed"]].iterrows():
            print(
                f"- {row['check_name']} on {row['column']} "
                f"({row['severity']} severity): {row['failed_rows']} failed rows"
            )


print_report_summary(validation_report)

## 14. Decide whether the pipeline should fail

In production, some checks should block a workflow and others should only create a warning.

A simple policy could be: if any high severity check fails, the validation fails.

In [ ]:
high_severity_failures = validation_report[
    (validation_report["severity"] == "high") & (~validation_report["passed"])
]

pipeline_validation_passed = high_severity_failures.empty

print(f"Pipeline validation passed: {pipeline_validation_passed}")
high_severity_failures

## Practice

Try these tasks:

1. Add a validation that checks `net_amount` is not null and greater than or equal to zero.
2. Change the accepted currencies to allow both `USD` and `EUR`.
3. Add a new check that validates row count is greater than zero.
4. Fix the data so all high severity checks pass.
5. Write a short incident-style note explaining the failed checks and the next investigation step.